In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "ethiopia"
vehicle = "salt"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
        .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
        .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention_25_nrv', 'intervention_100_nrv', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        )
        .assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,intervention_25_nrv,174,0,0
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,intervention_25_nrv,174,0,0
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,intervention_25_nrv,174,0,0
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,intervention_25_nrv,174,0,0
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,intervention_25_nrv,174,0,0
...,...,...,...,...,...,...,...,...,...,...,...
1067395,person_time,impairment,anemia,severe,95_plus,severe,1,baseline,121,0,0
1067396,person_time,impairment,anemia,severe,95_plus,severe,2,baseline,121,0,0
1067397,person_time,impairment,anemia,severe,95_plus,severe,3,baseline,121,0,0
1067398,person_time,impairment,anemia,severe,95_plus,severe,4,baseline,121,0,0


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline                200
intervention_100_nrv    200
intervention_25_nrv     200
zero                    200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

mild          360000
moderate      360000
not_anemic    360000
severe        360000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario              wealth_quintile
baseline              1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_100_nrv  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_25_nrv   1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
zero                  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario              wealth_quintile
baseline              1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_100_nrv  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_25_nrv   1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
zero                  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario              wealth_quintile
baseline              1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_100_nrv  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_25_nrv   1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
zero                  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,7729.330504
1,Female,0.0,0.019178,not_pregnant,2,7026.085890
2,Female,0.0,0.019178,not_pregnant,3,6642.401601
3,Female,0.0,0.019178,not_pregnant,4,5956.884779
4,Female,0.0,0.019178,not_pregnant,5,4774.758589
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,659.578610
281,Male,95.0,125.000000,not_pregnant,2,690.099645
282,Male,95.0,125.000000,not_pregnant,3,716.289850
283,Male,95.0,125.000000,not_pregnant,4,757.897451


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    831426.672018
2    844624.021487
3    692466.247031
4    643787.343974
5    642344.561183
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario              wealth_quintile
baseline              1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_100_nrv  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_25_nrv   1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
zero                  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        )
        .assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,intervention_25_nrv,174,0,0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,intervention_25_nrv,174,0,0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,intervention_25_nrv,174,0,0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,intervention_25_nrv,174,0,0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,intervention_25_nrv,174,0,0
...,...,...,...,...,...,...,...,...,...,...,...
533695,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,baseline,121,0,0
533696,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,baseline,121,0,0
533697,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,baseline,121,0,0
533698,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,baseline,121,0,0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['maternal_disorders_to_recovered_from_maternal_disorders',
       'no_transition',
       'susceptible_to_maternal_disorders_to_maternal_disorders'],
      dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario              wealth_quintile
baseline              1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_100_nrv  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_25_nrv   1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
zero                  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
Name: value, dtype: float64

In [18]:
path = f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"
if pathlib.Path(path).is_file():
    neonatal_deaths = pd.read_parquet(path).rename(columns={"maternal_scenario": "scenario"})
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,deaths,cause,stillborn,stillborn,0_to_6_months,Female,1,baseline,intervention_25_nrv,27,0,0
1,deaths,cause,stillborn,stillborn,0_to_6_months,Female,2,baseline,intervention_25_nrv,27,0,0
2,deaths,cause,stillborn,stillborn,0_to_6_months,Female,3,baseline,intervention_25_nrv,27,0,0
3,deaths,cause,stillborn,stillborn,0_to_6_months,Female,4,baseline,intervention_25_nrv,27,0,0
4,deaths,cause,stillborn,stillborn,0_to_6_months,Female,5,baseline,intervention_25_nrv,27,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
47915,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,baseline,40,0,0
47916,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,baseline,40,0,0
47917,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,baseline,40,0,0
47918,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,baseline,40,0,0


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario              wealth_quintile
baseline              1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_100_nrv  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_25_nrv   1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
zero                  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        )
        .assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,0,intervention_25_nrv
1,Female,0.0,0.019178,2,0,intervention_25_nrv
2,Female,0.0,0.019178,3,0,intervention_25_nrv
3,Female,0.0,0.019178,4,0,intervention_25_nrv
4,Female,0.0,0.019178,5,0,intervention_25_nrv
...,...,...,...,...,...,...
245,Male,95.0,125.000000,1,0,baseline
246,Male,95.0,125.000000,2,0,baseline
247,Male,95.0,125.000000,3,0,baseline
248,Male,95.0,125.000000,4,0,baseline


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario              wealth_quintile
baseline              1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_100_nrv  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_25_nrv   1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
zero                  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario              wealth_quintile
baseline              1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_100_nrv  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
intervention_25_nrv   1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
zero                  1                  0.0
                      2                  0.0
                      3                  0.0
                      4                  0.0
                      5                  0.0
Name: value, dtype: float64

In [25]:
path = (
    f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        )
        .assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario              wealth_quintile
zero                  1                  1760.034450
                      2                  1674.195341
                      3                  1523.810545
                      4                  1342.816454
                      5                  1059.347767
baseline              1                  1756.556323
                      2                  1670.886846
                      3                  1520.799236
                      4                  1340.162820
                      5                  1057.254316
intervention_25_nrv   1                   965.959424
                      2                   957.016468
                      3                   814.867992
                      4                  1022.020483
                      5                   860.833016
intervention_100_nrv  1                   326.702643
                      2                   339.090914
                      3                   267.862962
        

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)